# Final comparison

1. Validation accuracy of every model (the table for the presentation)
2. Do the models make the same mistakes?
3. The Kaggle file: for each task, the model with the best validation accuracy

Run the 4 model notebooks first. Each one saves its predictions in `<model>/outputs/`.

## 1. Setup

In [ ]:
import os
import sys

# The notebook is in a sub-folder. Go to the project folder (where common.py is).
if not os.path.exists("common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import itertools

import numpy as np
import pandas as pd

import common

MODELS = {                       # name -> folder
    "k-NN": "knn",
    "Decision Tree": "decision_tree",
    "Random Forest": "random_forest",
    "Neural Network": "neural_network",
}
TASKS = ["image", "sentiment"]
os.makedirs("final/submissions", exist_ok=True)

## 2. Load the validation predictions

Every model used the same validation rows (`common.stratified_split`), so we can compare them row by row.

In [ ]:
y_val = {}
for task in TASKS:
    _, y, _, _ = common.load_task(task)
    _, val_idx = common.stratified_split(y)
    y_val[task] = y[val_idx]

val_pred = {task: {} for task in TASKS}
for task in TASKS:
    for name, folder in MODELS.items():
        path = f"{folder}/outputs/{task}_val_pred.npy"
        if os.path.exists(path):
            val_pred[task][name] = np.load(path)
        else:
            print("missing:", path, "-> run that notebook first")

## 3. Validation accuracy of every model

In [ ]:
rows = []
for name in MODELS:
    row = {"Method": name}
    for task in TASKS:
        if name in val_pred[task]:
            row[f"{task} accuracy"] = round(common.accuracy(y_val[task], val_pred[task][name]), 4)
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv("final/validation_table.csv", index=False)
table

## 4. Do the models make the same mistakes?

For every pair of models: how many validation examples **both** get wrong, and how many only one of them gets wrong.

In [ ]:
for task in TASKS:
    wrong = {name: pred != y_val[task] for name, pred in val_pred[task].items()}
    print(f"\n{task} ({len(y_val[task])} validation examples)")
    for a, b in itertools.combinations(wrong, 2):
        both = (wrong[a] & wrong[b]).sum()
        only_a = (wrong[a] & ~wrong[b]).sum()
        only_b = (~wrong[a] & wrong[b]).sum()
        print(f"   {a} vs {b}: both wrong {both}, only {a} wrong {only_a}, only {b} wrong {only_b}")
    if wrong:
        all_wrong = np.logical_and.reduce(list(wrong.values())).sum()
        print(f"   wrong for every model: {all_wrong}")

## 5. Kaggle file

For each task we take the model with the best **validation** accuracy (not the Kaggle score), and use its test predictions.

In [ ]:
chosen = {}
for task in TASKS:
    scores = {name: common.accuracy(y_val[task], pred) for name, pred in val_pred[task].items()}
    chosen[task] = max(scores, key=scores.get)
    print(f"{task}: {chosen[task]} (validation accuracy {scores[chosen[task]]:.4f})")

image = pd.read_csv(f"{MODELS[chosen['image']]}/outputs/image_test_pred.csv")
sentiment = pd.read_csv(f"{MODELS[chosen['sentiment']]}/outputs/sentiment_test_pred.csv")

submission = common.make_submission(image["id"], image["label"], sentiment["id"], sentiment["label"],
                                    "final/submissions/submission_final.csv")
print(len(submission), "rows written to final/submissions/submission_final.csv")
submission.head()